# 第二讲 任务定义与数据生成

一行代表谁，预测发生在何时，答案如何得到？

请先阅读 lab.md 并完成纸笔判断。这里全部是原创合成数据，所有金额仅为教学数字。本实验不训练模型。你可以先使用 Restart Kernel 后 Run All，从头依次运行；暂时不需要理解每行 Python 语法。实验不访问网络。

## 1 核对文件

请把本文件与 experiment.py、data 文件夹放在同一个目录。下面只是检查文件是否存在，并加载已写好的审计步骤。True 表示存在，False 表示需要检查目录结构。

In [1]:
from pathlib import Path
from datetime import date
import experiment as exp

print("原始 CSV 存在：", (exp.BASE / "data" / "raw_events.csv").is_file())
print("标准答案 CSV 存在：", (exp.BASE / "data" / "expected_labels.csv").is_file())
raw = exp.read_rows(exp.BASE / "data" / "raw_events.csv")
print("原始导出行数：", len(raw))
assert len(raw) == 11

原始 CSV 存在： True
标准答案 CSV 存在： True
原始导出行数： 11


## 2 区分记录与事件

同一次挂牌 E01 被导出为 R01 和 R02；同一房屋 H01 的另一次挂牌 E07 不是副本。先预测下面会输出哪些编号，再运行。

In [2]:
events, duplicates = exp.deduplicate_events(raw)
print("完全副本：", duplicates)
print("不同事件数：", len(events))
print("不同房屋数：", len({r["house_id"] for r in events}))
print("H01 的不同挂牌：", [r["event_id"] for r in events if r["house_id"] == "H01"])
assert duplicates == [{"kept": "R01", "duplicate": "R02", "event_id": "E01"}]
assert len(events) == 10

完全副本： [{'kept': 'R01', 'duplicate': 'R02', 'event_id': 'E01'}]
不同事件数： 10
不同房屋数： 9
H01 的不同挂牌： ['E01', 'E07']


## 3 截止日期与标签

规则已经写在讲义中：窗口含第 30 天；正例需要已登记的窗口内成交；负例需要完整核实窗口；其余未知。输出的 None 是未知，不是零。

In [3]:
labels = {r["event_id"]: exp.make_label(r) for r in events}
for event, label in labels.items():
    print(event, "未知" if label is None else label)
print("去重后汇总：", exp.known_summary(list(labels.values())))
print("去重前汇总：", exp.known_summary([exp.make_label(r) for r in raw]))
assert labels["E04"] == 0
assert labels["E08"] is None and labels["E09"] is None
assert labels["E10"] == 1

E01 1
E02 1
E03 0
E04 0
E05 1
E06 0
E07 1
E08 未知
E09 未知
E10 1
去重后汇总： {'known': 8, 'positive': 5, 'negative': 3, 'unknown': 2, 'positive_fraction': 0.625}
去重前汇总： {'known': 9, 'positive': 6, 'negative': 3, 'unknown': 2, 'positive_fraction': 0.6666666666666666}


## 4 检查挂牌时输入资格

下面分开显示可用、未来与缺失。面积晚一天到达也不符合预测时点；已有未来数值不代表当时能读到。

In [4]:
audit = exp.feature_audit(events)
for field, result in audit.items():
    print(field)
    print("  可用：", result["eligible"])
    print("  未来：", result["future"])
    print("  缺失：", result["missing"])
assert audit["area_sqm"]["future"] == ["E06"]
assert len(audit["views_30d"]["future"]) == 10

area_sqm
  可用： ['E01', 'E02', 'E03', 'E04', 'E05', 'E07', 'E08', 'E09', 'E10']
  未来： ['E06']
  缺失： []
list_price_wan
  可用： ['E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09', 'E10']
  未来： []
  缺失： []
views_30d
  可用： []
  未来： ['E01', 'E02', 'E03', 'E04', 'E05', 'E06', 'E07', 'E08', 'E09', 'E10']
  缺失： []
fee_wan
  可用： []
  未来： ['E01', 'E02', 'E04', 'E05', 'E07', 'E09', 'E10']
  缺失： ['E03', 'E06', 'E08']


## 5 两种对象划分

相同事件不重叠仍可能共享房屋、楼栋。请对照手写的集合交集。按楼栋隔离只是匹配一种任务要求，不保证样本量足够或时间设计正确。

In [5]:
bad_train = {"E01", "E02", "E03", "E05"}
bad_test = {r["event_id"] for r in events} - bad_train
for key in ["event_id", "house_id", "building_id"]:
    print("坏划分", key, "交集：", exp.overlap(events, bad_train, bad_test, key))

group_train = {r["event_id"] for r in events if r["building_id"] in {"B01", "B02", "B03"}}
group_test = {r["event_id"] for r in events} - group_train
print("按楼栋隔离的测试事件：", sorted(group_test))
print("楼栋交集：", exp.overlap(events, group_train, group_test, "building_id"))
print("测试侧已知标签数：", sum(labels[e] is not None for e in group_test))

坏划分 event_id 交集： []
坏划分 house_id 交集： ['H01']
坏划分 building_id 交集： ['B01', 'B02', 'B03']
按楼栋隔离的测试事件： ['E08', 'E09', 'E10']
楼栋交集： []
测试侧已知标签数： 1


## 6 改变已知时间 不改变历史事实

E09 的成交在 4 月 22 日才登记。将资料截止日从 4 月 20 日改成 4 月 23 日，改变的是能知道的内容，不是已经发生的成交日。E08 缺少后续完整核实，单纯把截止日期变晚不会补出证据。

In [6]:
e09 = next(r for r in events if r["event_id"] == "E09")
e08 = next(r for r in events if r["event_id"] == "E08")
print("E09 截止 4 月 20 日：", exp.make_label(e09, date(2025, 4, 20)))
print("E09 截止 4 月 23 日：", exp.make_label(e09, date(2025, 4, 23)))
print("E08 仅把截止推迟到 6 月 1 日：", exp.make_label(e08, date(2025, 6, 1)))
assert exp.make_label(e09, date(2025, 4, 23)) == 1
assert exp.make_label(e08, date(2025, 6, 1)) is None

E09 截止 4 月 20 日： None
E09 截止 4 月 23 日： 1
E08 仅把截止推迟到 6 月 1 日： None


## 7 边界输入与错误拦截

检查第 0、30、31 天，延迟登记，随访不足，核实证据尚未可用，冲突版本，以及没有已知标签的情形。异常被预期地拦截也是一种正确结果。

In [7]:
checks = exp.boundary_tests(events[0])
print("边界检查通过：", len(checks))
for check in checks:
    print("通过", check)

边界检查通过： 9
通过 day_30_included
通过 day_31_excluded
通过 short_followup_is_unknown
通过 verification_not_yet_available
通过 delayed_record_changes_knowledge
通过 day_0_included
通过 prediction_not_yet_occurred
通过 conflicting_duplicate_rejected
通过 no_known_labels_no_division_by_zero


## 8 完整报告与一致性

下面重新走一遍全部步骤，并输出 JSON 与 CSV 报告。断言比较它与前面的逐步结果。完美的费用算术是非法信息捷径，不是可部署模型的性能。

In [8]:
report = exp.run_experiment(write_outputs=True)
exp.print_report(report)
print("3 月 1 日冻结的时间划分：", report["temporal_split"])
assert report["labels"] == labels
assert report["feature_audit"] == audit
assert report["boundary_tests"] == checks
print("脚本与逐步核对一致：", True)
print("报告文件存在：", (exp.BASE / "outputs" / "audit_report.json").is_file())

原始行 / 不同事件 / 不同房屋 / 不同楼栋： 11 10 9 5
重复副本：R02 与 R01 指向同一次 E01
逐事件标签： {'E01': 1, 'E02': 1, 'E03': 0, 'E04': 0, 'E05': 1, 'E06': 0, 'E07': 1, 'E08': None, 'E09': None, 'E10': 1}
去重后已知正例比例：5 / 8 = 62.50%
去重前已知正例比例：6 / 9 = 66.67%
挂牌时不可用的面积： ['E06']
未来浏览量事件数： 10
未来费用事件数： 7
坏划分房屋交集： ['H01']
坏划分楼栋交集： ['B01', 'B02', 'B03']
按楼栋隔离后的楼栋交集： []
按楼栋隔离后测试侧已知标签数： 1
费用捷径 MAE 为 0，但 valid_prediction = False
边界检查通过数： 9
所有预期核对通过。这里只验证合成数据审计，不验证现实预测能力。
3 月 1 日冻结的时间划分： {'freeze_date': '2025-03-01', 'historical_known': ['E01', 'E02', 'E03', 'E04', 'E05', 'E06'], 'later_events': ['E07', 'E08', 'E09', 'E10']}
脚本与逐步核对一致： True
报告文件存在： True


## 你的结论

另存一份后，在本单元写出：样本单位、标签规则、你发现的副本与未来输入、分组重叠，以及这个小实验不能证明什么。不能只写运行通过。

制作验证说明：此本已在新的 Python 进程中由真实 ipykernel 进程内内核顺序执行，保留输出。构建环境禁止套接字，未验证浏览器 Jupyter UI 或外进程内核通信；离线计算不依赖这些界面。